In [ ]:
!pip install fastapi uvicorn nest_asyncio

In [1]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request
import asyncio
from datetime import datetime

# 1. Apply fix to allow nested event loops in Jupyter
nest_asyncio.apply()

# 2. Initialize Gateway
app = FastAPI(title="Digital Economy Data Security Gateway (Jupyter Background Version)")

# --- Mock Database and Complex Policies ---

USERS = {
    "token_alice_123": {
        "corp": "Alice_Corp",
        "clearance_level": 3,
        "allowed_ips": ["127.0.0.1", "192.168.1.100"],
        "daily_quota": 5,
        "usage_today": 1
    }
}

DATASETS = {
    "data_finance_01": {
        "filename": "UK_Consumer_Spending_2025.csv",
        "sensitivity_level": 2,
        "region": "UK"
    },
    "data_core_02": {
        "filename": "Global_Trade_Secrets.csv",
        "sensitivity_level": 5,
        "region": "GLOBAL"
    }
}

POLICIES = {
    "Alice_Corp": {
        "allowed_regions": ["UK", "EU", "US"],
        "business_hours": {"start": 8, "end": 18}, # Access allowed between 08:00 and 18:00
        "watermark_profile": {
            "require_watermark": True, 
            "buyer_signature": "ALICE_8871",
            "encryption_algo": "AES-256-GCM"
        }
    }
}

# --- Core Interception Endpoint ---
@app.get("/api/v1/secure-download/{dataset_id}")
def download_data(dataset_id: str, request: Request, x_token: str = Header(None)):
    # Layer 1: Basic Token Validation
    if not x_token or x_token not in USERS:
        raise HTTPException(status_code=401, detail="Gateway Intercept: Missing or invalid Token.")
    
    user_info = USERS[x_token]
    corp_name = user_info["corp"]
    policy = POLICIES.get(corp_name)
    
    # Layer 2: IP Whitelist Validation
    client_ip = request.client.host
    if client_ip not in user_info["allowed_ips"]:
        raise HTTPException(status_code=403, detail=f"Gateway Intercept: Unauthorized IP address ({client_ip}).")

    # Layer 3: Time Window Validation (based on server local time)
    current_hour = datetime.now().hour
    if not (policy["business_hours"]["start"] <= current_hour < policy["business_hours"]["end"]):
        raise HTTPException(status_code=403, detail="Gateway Intercept: Current time is outside of policy-allowed business hours.")

    # Layer 4: Quota Validation
    if user_info["usage_today"] >= user_info["daily_quota"]:
        raise HTTPException(status_code=429, detail="Gateway Intercept: Daily data access quota exceeded.")

    # Layer 5: Dataset Existence Validation
    if dataset_id not in DATASETS:
        raise HTTPException(status_code=404, detail="Gateway Intercept: Requested dataset does not exist.")
    
    dataset_info = DATASETS[dataset_id]

    # Layer 6: Data Clearance/Sensitivity Validation
    if user_info["clearance_level"] < dataset_info["sensitivity_level"]:
        raise HTTPException(status_code=403, detail="Gateway Intercept: Insufficient security clearance for this highly sensitive data.")

    # Layer 7: Regional Compliance Validation
    if dataset_info["region"] not in policy["allowed_regions"]:
        raise HTTPException(status_code=403, detail=f"Gateway Intercept: Cross-border policy restriction. Access to region '{dataset_info['region']}' denied.")
        
    # All checks passed, simulate quota deduction
    user_info["usage_today"] += 1

    return {
        "status": "SUCCESS",
        "corp": corp_name,
        "dataset_name": dataset_info["filename"],
        "message": "Gateway cleared. Applying security policies...",
        "security_measures": {
            "watermark_injected": policy["watermark_profile"]["buyer_signature"],
            "encryption_applied": policy["watermark_profile"]["encryption_algo"],
            "quota_remaining": user_info["daily_quota"] - user_info["usage_today"]
        }
    }

# ==========================================
# 3. Start Background Task
# ==========================================
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

asyncio.create_task(server.serve())

print("Gateway successfully started silently in the background!")
print("View API Documentation: http://127.0.0.1:8000/docs")
print("Test Interception: http://127.0.0.1:8000/api/v1/secure-download/data_finance_01")

Gateway successfully started silently in the background!
View API Documentation: http://127.0.0.1:8000/docs
Test Interception: http://127.0.0.1:8000/api/v1/secure-download/data_finance_01


INFO:     Started server process [16360]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51680 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51680 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:61142 - "GET /api/v1/secure-download/data_finance_01 HTTP/1.1" 200 OK


In [ ]:
#真实的网关会使用 JWT (JSON Web Token)。JWT 是一串经过加密签名的长字符串，里面自带了过期时间（Expiration）。
#就算黑客偷走了 Token，几个小时后它也就自动作废了。而且网关可以通过验证签名，确保 Token 没有被篡改过。

#必须加入日志留存机制。每一笔请求（无论成功还是被拦截）都要详细记录下来，通常会写入专门的日志中心（如 Elasticsearch）。
#记录内容： 时间戳、真实 IP、请求的用户、操作的数据集、成功/失败状态、失败原因。

#全面异步化： 在查询数据库或调用其他内部接口时，使用 async/await，这样网关在等数据库返回结果时，就不会卡住去处理其他人的请求，极大提升吞吐量。
#使用 Pydantic 模型： FastAPI 的绝配。用 Pydantic 定义好进出网关的数据格式，这样 FastAPI 会自动帮你做数据类型校验，并且在 /docs 文档里生成更完美的接口说明。